# PREP_01 - Patient Split

## Objective

Generate a reproducible patient-level split for all downstream experiments.

The exploratory analysis performed in EDA_01.ipynb demonstrated that multiple lesions belong to the same patient.

Therefore, image-level random splitting would introduce data leakage.

This notebook creates a train, validation and test partition at the patient level.

In [36]:
# import importlib
import pandas as pd
import numpy as np

# from sklearn.model_selection import StratifiedGroupKFold

from skin_lesion_ai.utils.data_utils import load_metadata_parquet, save_metadata_parquet

## Load preprocessed_from_raw.parquet

In [37]:
# Load the latest preprocessed metadata parquet file

# latest_file = get_latest_metadata_parquet(
#     prefix="preprocessed_from_raw",
#     stage = "interim"
# )

# df = load_metadata_parquet(
#     stage="interim",
#     filename=latest_file
# )

df = load_metadata_parquet(
    stage="interim",
    filename="preprocessed_from_raw",
    timestamp_flag=True,
)

# print(latest_file)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 401059 entries, 0 to 401058
Data columns (total 13 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   isic_id                       401059 non-null  str    
 1   patient_id                    401059 non-null  str    
 2   attribution                   401059 non-null  str    
 3   copyright_license             401059 non-null  str    
 4   age_approx                    398261 non-null  float64
 5   sex                           389542 non-null  str    
 6   anatom_site_general           395303 non-null  str    
 7   clin_size_long_diam_mm        401059 non-null  float64
 8   tbp_lv_nevi_confidence        401059 non-null  float64
 9   tbp_lv_dnn_lesion_confidence  401059 non-null  float64
 10  tbp_lv_location               401059 non-null  str    
 11  tbp_lv_location_simple        401059 non-null  str    
 12  diagnostic_group              401059 non-null  string 


In [38]:
# View the shape and first few rows of the dataframe

print(df.shape)
df.head()

(401059, 13)


,isic_id,patient_id,attribution,copyright_license,age_approx,sex,anatom_site_general,clin_size_long_diam_mm,tbp_lv_nevi_confidence,tbp_lv_dnn_lesion_confidence,tbp_lv_location,tbp_lv_location_simple,diagnostic_group
0,ISIC_0015670,IP_1235828,Memorial Sloan Kettering Cancer Center,CC-BY,60.0,male,lower extremity,3.04,2.628592e-03,97.517282,Right Leg - Upper,Right Leg,benign_non_biopsied
1,ISIC_0015845,IP_8170065,Memorial Sloan Kettering Cancer Center,CC-BY,60.0,male,head/neck,1.10,1.334303e-07,3.141455,Head & Neck,Head & Neck,benign_non_biopsied
2,ISIC_0015864,IP_6724798,Memorial Sloan Kettering Cancer Center,CC-BY,60.0,male,posterior torso,3.40,2.959180e-04,99.804040,Torso Back Top Third,Torso Back,benign_non_biopsied
3,ISIC_0015902,IP_4111386,ACEMID MIA,CC-0,65.0,male,anterior torso,3.22,2.198945e+01,99.989998,Torso Front Top Half,Torso Front,benign_non_biopsied
4,ISIC_0024200,IP_8313778,Memorial Sloan Kettering Cancer Center,CC-BY,55.0,male,anterior torso,2.73,1.378832e-03,70.442510,Torso Front Top Half,Torso Front,benign_non_biopsied


In [39]:
# Check how many unique patients we have in the dataset

df["patient_id"].nunique()

1042

In [40]:
# Check how many samples we have per diagnostic group

df["diagnostic_group"].value_counts()

diagnostic_group
benign_non_biopsied       399991
benign_biopsied              561
malignant_biopsied           393
indeterminate_biopsied       114
Name: count, dtype: int64[pyarrow]

In [41]:
# Check how many samples we have per patient
# Count the number of samples per patient and then describe the distribution

df.groupby("patient_id").size().describe()  # size() instead of count()

count    1042.000000
mean      384.893474
std       540.268913
min         1.000000
25%       115.000000
50%       241.500000
75%       477.500000
max      9184.000000
dtype: float64

In [42]:
# Create a patient-level dataframe with the following columns:
# - patient_id
# - n_lesions: number of lesions for that patient
# - has_biopsied_lesion: whether the patient has at least one biopsied lesion (True/False)
# - has_malignant_lesion: whether the patient has at least one malignant lesion

patient_level_df = (
    df.groupby("patient_id")
    .agg(
        n_img_lesions=("isic_id", "count"),
        has_biopsied_lesion=(
            "diagnostic_group",
            lambda x: x.isin(
                ["benign_biopsied", "indeterminate_biopsied", "malignant_biopsied"]
            ).any(),
        ),
        has_malignant_lesion=(
            "diagnostic_group",
            lambda x: (x == "malignant_biopsied").any(),
        ),
    )
    .reset_index()
)

In [43]:
# Describe the patient-level dataframe

patient_level_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1042 entries, 0 to 1041
Data columns (total 4 columns):
 #   Column                Non-Null Count  Dtype        
---  ------                --------------  -----        
 0   patient_id            1042 non-null   str          
 1   n_img_lesions         1042 non-null   int64        
 2   has_biopsied_lesion   1042 non-null   bool[pyarrow]
 3   has_malignant_lesion  1042 non-null   bool[pyarrow]
dtypes: bool[pyarrow](2), int64(1), str(1)
memory usage: 27.0 KB


# Create Stratification Target

The primary modelling objective of the project is biopsy recommendation.

To preserve clinically relevant cases across data partitions, patient-level stratification is based on whether a patient has at least one biopsied lesion.

This target will be used to generate reproducible train, validation and test splits.

In [44]:
# View the first few rows of the patient-level dataframe

patient_level_df.head()

,patient_id,n_img_lesions,has_biopsied_lesion,has_malignant_lesion
0,IP_0008821,102,False,False
1,IP_0014998,352,True,False
2,IP_0023586,939,True,True
3,IP_0028775,212,True,False
4,IP_0028993,225,True,False


In [45]:
# Check the distribution of the has_biopsied_lesion column

patient_level_df["has_biopsied_lesion"].value_counts()

has_biopsied_lesion
True     660
False    382
Name: count, dtype: int64[pyarrow]

In [46]:
# Check the distribution of the has_malignant_lesion column

patient_level_df["has_malignant_lesion"].value_counts()

has_malignant_lesion
False    783
True     259
Name: count, dtype: int64[pyarrow]

## Split configuration

The dataset is partitioned at patient level using:

- Train: 80%
- Validation: 10%
- Test: 10%

Stratification is performed using the variable
`has_biopsied_lesion`.

A fixed random seed (42) is used to ensure
reproducibility.

In [47]:
# Split the patient-level dataframe into train, validation, and test sets
# using stratified sampling based on the has_biopsied_lesion column

from sklearn.model_selection import train_test_split

TRAIN_SIZE = 0.80
VAL_SIZE = 0.10
TEST_SIZE = 0.10
RANDOM_STATE = 42

assert np.isclose(TRAIN_SIZE + VAL_SIZE + TEST_SIZE, 1.0)

In [48]:
# Calculate the size of the holdout set (validation + test)

holdout_size = VAL_SIZE + TEST_SIZE

# We will first split the data into a training set and a holdout set

train_patients, holdout_patients = train_test_split(
    patient_level_df,
    test_size=holdout_size,
    stratify=patient_level_df["has_biopsied_lesion"],
    random_state=RANDOM_STATE,
)

In [49]:
# Now we need to split the holdout set into validation and test sets.

relative_test_size = TEST_SIZE / (VAL_SIZE + TEST_SIZE)

val_patients, test_patients = train_test_split(
    holdout_patients,
    test_size=relative_test_size,
    stratify=holdout_patients["has_biopsied_lesion"],
    random_state=RANDOM_STATE,
)

relative_test_size

0.5

## Asignar split

In [50]:
# assign the split labels to the patient-level dataframes

train_patients["split"] = "train"
val_patients["split"] = "val"
test_patients["split"] = "test"

In [53]:
# Combine the patient-level dataframes back into a single dataframe

patient_split_df = pd.concat(
    [train_patients, val_patients, test_patients], ignore_index=True
)

patient_split_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1042 entries, 0 to 1041
Data columns (total 5 columns):
 #   Column                Non-Null Count  Dtype        
---  ------                --------------  -----        
 0   patient_id            1042 non-null   str          
 1   n_img_lesions         1042 non-null   int64        
 2   has_biopsied_lesion   1042 non-null   bool[pyarrow]
 3   has_malignant_lesion  1042 non-null   bool[pyarrow]
 4   split                 1042 non-null   str          
dtypes: bool[pyarrow](2), int64(1), str(2)
memory usage: 40.0 KB


In [54]:
# patient_split_df is the final patient-level dataframe with the split labels assigned.
# We can now save this dataframe as a parquet file for later use.

patient_split_df.head()

,patient_id,n_img_lesions,has_biopsied_lesion,has_malignant_lesion,split
0,IP_2482185,273,False,False,train
1,IP_6308791,142,True,False,train
2,IP_9225429,119,False,False,train
3,IP_8721252,595,True,True,train
4,IP_8861604,129,False,False,train


## Validación de distribución

In [55]:
# Check the distribution of the split labels

patient_split_df["split"].value_counts()

split
train    833
test     105
val      104
Name: count, dtype: int64

In [56]:
# Check the distribution of the split labels as percentages

patient_split_df["split"].value_counts(normalize=True)

split
train    0.799424
test     0.100768
val      0.099808
Name: proportion, dtype: float64

In [57]:
# Check the distribution of the has_biopsied_lesion column within each split

patient_split_df.groupby("split")["has_biopsied_lesion"].mean()

split
test     0.628571
train    0.633854
val      0.634615
Name: has_biopsied_lesion, dtype: double[pyarrow]

In [58]:
# Check the distribution of the has_malignant_lesion column within each split

patient_split_df.groupby("split")["has_malignant_lesion"].mean()

split
test     0.285714
train    0.243697
val          0.25
Name: has_malignant_lesion, dtype: double[pyarrow]

In [59]:
# Check the number of unique patients in each split to confirm that there is no overlap

patient_split_df.groupby("split")["patient_id"].nunique()

split
test     105
train    833
val      104
Name: patient_id, dtype: int64

## Data Leakage

In [60]:
# Check that there is no patient overlap between the splits

assert len(set(train_patients["patient_id"]) & set(val_patients["patient_id"])) == 0
assert len(set(train_patients["patient_id"]) & set(test_patients["patient_id"])) == 0
assert len(set(val_patients["patient_id"]) & set(test_patients["patient_id"])) == 0

In [61]:
# Merge the split labels back to the original lesion-level dataframe

df_split = df.merge(
    patient_split_df[["patient_id", "split"]], on="patient_id", how="inner"
)

df_split.groupby("split")["isic_id"].count()

split
test      51070
train    311943
val       38046
Name: isic_id, dtype: int64

In [62]:
# Check the distribution of diagnostic groups within each split
# to confirm that the stratification worked as intended

(df_split.query("diagnostic_group != 'benign_non_biopsied'").groupby("split").size())

split
test      98
train    862
val      108
dtype: int64

In [63]:
# Check the number of unique patients in each split to confirm that there is no overlap

df_split.groupby("split")["patient_id"].nunique()

split
test     105
train    833
val      104
Name: patient_id, dtype: int64

In [64]:
# Final assertion to confirm that the number of unique patients in the original
# dataframe matches the number of unique patients in the patient-level split dataframe

assert df_split["patient_id"].nunique() == patient_split_df["patient_id"].nunique()

# Persistencia

In [65]:
# Save the patient-level dataframe with split labels as a parquet file for later use

output_path = save_metadata_parquet(
    patient_split_df, stage="interim", name="patient_split", timestamp=True
)

print(output_path)

D:\Workspace\research\ub_master_thesis\my-image-classifier\data\interim\metadata\patient_split_20260617_195834.parquet


# Findings and Conclusions

## Patient-level partitioning

The exploratory analysis demonstrated that multiple lesions belong to the same patient. Therefore, image-level random splitting would introduce data leakage and artificially inflate model performance.

To avoid this issue, the dataset was partitioned at the patient level.

A total of 1,042 unique patients were identified and assigned to one of three mutually exclusive subsets:

* Training: 833 patients (79.94%)
* Validation: 104 patients (9.98%)
* Test: 105 patients (10.08%)

No patient appears in more than one subset.

---

## Stratification results

Patient-level stratification was performed using the variable `has_biopsied_lesion`, which represents the primary modelling objective of the project.

The prevalence of patients with at least one biopsied lesion was successfully preserved across all partitions:

| Split      | Biopsied prevalence |
| ---------- | ------------------: |
| Train      |              63.39% |
| Validation |              63.46% |
| Test       |              62.86% |

This confirms that the split procedure maintained a consistent distribution of clinically relevant cases.

---

## Malignant patient distribution

Although malignancy was not used as the stratification target, the prevalence of patients with at least one malignant lesion remained reasonably balanced across partitions:

| Split      | Malignant prevalence |
| ---------- | -------------------: |
| Train      |               24.37% |
| Validation |               25.00% |
| Test       |               28.57% |

This distribution is considered acceptable for downstream modelling.

---

## Lesion-level implications

The patient-level split results in different numbers of lesions per subset because patients contribute highly variable numbers of lesions.

After propagating split labels back to the lesion-level dataset:

* Train: 311,943 lesions
* Validation: 38,046 lesions
* Test: 51,070 lesions

This imbalance is expected and reflects the natural distribution of lesions across patients.

---

## Biopsied lesions

The complete biopsied subset contains 1,068 lesions:

* Benign biopsied: 561
* Indeterminate biopsied: 114
* Malignant biopsied: 393

These lesions were distributed across train, validation and test subsets through the patient-level partitioning strategy, ensuring that no patient-level information leaks between datasets.

---

## Output artefact

The notebook generates a single reproducible patient-level partition file:

`patient_split.parquet`

This file becomes the master split definition for all subsequent modelling stages:

* H1: Biopsy recommendation model
* H2: Malignancy classification model
* H3: Multimodal image + metadata model

All future datasets and experiments must use this partition file to guarantee consistency and reproducibility throughout the project.
